# 69장 · FastAPI StreamingResponse와 줄 단위 JSON

스트리밍 응답은 본문을 여러 조각으로 전달합니다. 이 예제는 줄바꿈으로 JSON 이벤트를 나누고 완료 이벤트를 따로 보냅니다.

답이 화면에 한 조각 보인 뒤 연결이 끊겼습니다. 사용자는 답을 다 받은 것일까요? 전체 서비스는 검증을 마친 답을 NDJSON으로 나누어 보냅니다. 노트북에서는 고정한 답을 사용해 일부만 받았을 때와 저장 뒤 완료 이벤트를 받았을 때를 구분합니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 fastapi 0.135.0입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## StreamingResponse의 이벤트 직렬화

NDJSON에서는 JSON 객체 하나 뒤에 줄바꿈을 둡니다. 실제 네트워크 조각의 끝이 줄의 끝과 일치하지 않을 수 있으므로, 클라이언트는 완성된 줄부터 읽습니다. 아래는 /conversations/{cid}/stream 경로의 줄 단위 전송과 소유자 검사를 축약한 로컬 앱입니다.

### 스트림의 시작과 마지막 줄 구별하기

**스트림을 열었습니다.**　시작 이벤트에는 답변이 끝났다는 정보가 없습니다.

```
받은 이벤트: start
```

**이 예에서는 답 한 문장이 한 번에 옵니다.**　화면에 문장이 다 보여도 done을 받기 전에는 수신 측에서 저장 여부와 완료 여부를 확인할 수 없습니다.

```
받은 이벤트: start, delta
```

**완료 이벤트를 받았습니다.**　노트북이 정상 응답을 마지막 줄까지 읽었을 때 확인한 순서입니다. 서버의 대화 기록에 저장된 항목 수도 셀에서 따로 확인합니다.

```
받은 이벤트: start, delta, done
```

**전송 조각과 이벤트 경계**　긴 띠를 나눈 전송 조각과 수신기가 구분하는 메시지를 비교하세요. 이 실습에서 이벤트 하나가 끝나는 위치는 JSON 한 줄의 줄바꿈입니다.

[웹 교재에서 열기](https://langchain-lecture.vercel.app/chapter69/index.html#concept-clip)


### 셀 01 · StreamingResponse 경로와 소유자 검사

start가 보낸 요청 ID를 보관합니다. 이 예제의 delta는 이미 만든 답을 나눈 조각입니다.


In [1]:
import json
from fastapi import FastAPI, Header, HTTPException
from fastapi.responses import StreamingResponse
from fastapi.testclient import TestClient

app = FastAPI()
owners = {"chat-a": "course-alice"}
history = {"chat-a": []}
def line(kind, **fields):
    return json.dumps({"type": kind, **fields}, ensure_ascii=False) + "\n"

@app.post("/conversations/{cid}/stream")
def stream(cid: str, body: dict, x_course_token: str = Header()):
    if owners.get(cid) != x_course_token:
        raise HTTPException(404)
    def events():
        yield line("start", request_id="demo-1", llm_token_stream=False)
        answer = "과제함에 제출하세요."
        yield line("delta", text=answer)
        history[cid].append((body["question"], answer))
        yield line("done", answer=answer)
    return StreamingResponse(events(), media_type="application/x-ndjson")

client = TestClient(app)

여기서는 공개 수업용 토큰을 소유자 구분에만 씁니다. 실제 로그인, 외부 모델 토큰 스트리밍, 네트워크 재접속은 구현하지 않았습니다.


### 셀 02 · 줄 단위 JSON과 완료 이벤트

질문 한 건을 보내고 이벤트 종류를 받은 순서대로 확인하세요.


In [2]:
with client.stream("POST", "/conversations/chat-a/stream",
                   headers={"X-Course-Token": "course-alice"},
                   json={"question": "과제는 어디에 내나요?"}) as response:
    events = [json.loads(record) for record in response.iter_lines()]
print([event["type"] for event in events])
print(len(history["chat-a"]))

['start', 'delta', 'done']
1


start, delta, done 뒤에 이력 수 1이 보입니다. done을 내보내기 직전에 이력을 저장했습니다. TestClient에서는 실제 휴대폰의 전송 간격이나 중단 동작을 측정하지 않습니다.


## 수신한 본문과 완료 이벤트 구분

저장 뒤 발송한 완료 신호를 클라이언트가 받지 못할 수도 있습니다. 문장이 마침표로 끝났다는 이유만으로 저장 여부를 확정하면 안 됩니다.


### 과제 03 · 부분 스트림의 화면 상태

수신 기록의 마지막 done이 빠졌습니다. 화면이 완료를 확인할 수 있는지 검사하세요.

먼저 끝까지 받은 events에서 마지막을 잘라 수신자가 본 앞부분만 재현합니다. 여기서 완료 신호를 못 봤다고 해서 서버 저장도 실패했다고 결론 내릴 수는 없습니다. 이 노트북에서는 history의 저장 기록을 따로 확인하세요.


In [3]:
partial = events[:2]
# 이벤트 목록에 done이 있는지 검사하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>partial = events[:2]
print(any(item[&#x27;type&#x27;] == &#x27;done&#x27; for item in partial))</code></pre>
<p>실행 결과</p><pre><code>False</code></pre>
<p>False입니다. 이는 클라이언트가 완료 신호를 보지 못했다는 뜻입니다. 서버 이력의 저장 여부는 대화를 다시 조회해야 확인할 수 있습니다.</p>
</details>


### 과제 04 · 다른 소유자의 조회 제한

같은 경로에 course-bob으로 요청하고 HTTP 코드를 확인하세요.


In [4]:
# chat-a 경로에 bob의 수업용 토큰으로 요청하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>other = client.post(&#x27;/conversations/chat-a/stream&#x27;, headers={&#x27;X-Course-Token&#x27;: &#x27;course-bob&#x27;}, json={&#x27;question&#x27;: &#x27;제출 위치&#x27;})
print(other.status_code)</code></pre>
<p>실행 결과</p><pre><code>404</code></pre>
<p>404입니다. 다른 사용자의 대화 존재 여부나 내용을 응답에 포함하지 않았습니다.</p>
</details>


## 돌아보기

스트림에 출력된 답, 서버에 저장된 턴, 화면이 확인한 완료 신호를 따로 기록하세요. 70장에서는 같은 저장 규칙을 문서와 데이터베이스로 이어갑니다.
